In [3]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
#Configurações do projeto

RAIZ_PROJETO = Path.cwd()

if RAIZ_PROJETO.name == "notebooks":
    RAIZ_PROJETO = RAIZ_PROJETO.parent

CAMINHO_DADOS = RAIZ_PROJETO / "dados"
CAMINHO_DADOS.mkdir(parents=True, exist_ok=True)

CAMINHO_BANCO = CAMINHO_DADOS / "agro_leads.db"

QUANTIDADE_LEADS = 950_000
TAMANHO_LOTE = 100_000
SEMENTE_ALEATORIA = 42

gerador = np.random.default_rng(SEMENTE_ALEATORIA)

print("Raiz do projeto:", RAIZ_PROJETO)
print("Banco SQLite:", CAMINHO_BANCO)

Raiz do projeto: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\agro-leads-orchestrator
Banco SQLite: c:\Users\DataCore\Desktop\projetos_Git\agro-leads-orchestrator\agro-leads-orchestrator\dados\agro_leads.db


In [5]:
#Criar conexão com SQLite

def criar_conexao(caminho_banco: Path) -> sqlite3.Connection:
    conexao = sqlite3.connect(caminho_banco)

    conexao.execute("PRAGMA journal_mode = WAL;")
    conexao.execute("PRAGMA synchronous = NORMAL;")
    conexao.execute("PRAGMA temp_store = MEMORY;")
    conexao.execute("PRAGMA cache_size = -200000;")
    conexao.execute("PRAGMA foreign_keys = ON;")

    return conexao

In [6]:
#Remover banco antigo, se existir

if CAMINHO_BANCO.exists():
    CAMINHO_BANCO.unlink()
    print("Banco anterior removido.")

conexao = criar_conexao(CAMINHO_BANCO)
print("Conexão criada com sucesso.")

Banco anterior removido.
Conexão criada com sucesso.


In [7]:
#Criar tabela leads

def criar_tabela_leads(conexao: sqlite3.Connection) -> None:
    consulta_sql = """
    CREATE TABLE leads (
        id_cliente INTEGER PRIMARY KEY,
        nome TEXT NOT NULL,
        telefone TEXT NOT NULL UNIQUE,

        cultura TEXT NOT NULL CHECK (
            cultura IN ('Cana', 'Soja', 'Milho')
        ),

        estagio_atual TEXT NOT NULL CHECK (
            estagio_atual IN (
                'Plantio',
                'Desenvolvimento',
                'Safra',
                'Entresafra'
            )
        ),

        status_atual TEXT NOT NULL CHECK (
            status_atual IN (
                'Disponível',
                'Em Cooldown',
                'Fila Prioritária',
                'Em Atendimento',
                'Convertido'
            )
        ),

        ultimo_contato TEXT,
        cooldown_ate TEXT,
        score_prioridade REAL NOT NULL
    );
    """

    conexao.execute(consulta_sql)
    conexao.commit()

    print("Tabela leads criada.")

In [8]:
#Criar tabela eventos_contato

def criar_tabela_eventos_contato(conexao: sqlite3.Connection) -> None:
    consulta_sql = """
    CREATE TABLE eventos_contato (
        id_evento INTEGER PRIMARY KEY AUTOINCREMENT,
        id_cliente INTEGER NOT NULL,

        data_evento TEXT NOT NULL,

        canal TEXT NOT NULL CHECK (
            canal IN ('Robô', 'Humano', 'WhatsApp', 'Sistema')
        ),

        resultado TEXT NOT NULL CHECK (
            resultado IN (
                'Atendido',
                'Não Atendido',
                'Venda',
                'Resposta WhatsApp',
                'Transferência Assistida',
                'Cooldown Aplicado',
                'Lead Gerado'
            )
        ),

        observacao TEXT,

        FOREIGN KEY (id_cliente) REFERENCES leads(id_cliente)
    );
    """

    conexao.execute(consulta_sql)
    conexao.commit()

    print("Tabela eventos_contato criada.")

In [9]:
#Executar criação das tabelas

criar_tabela_leads(conexao)
criar_tabela_eventos_contato(conexao)

Tabela leads criada.
Tabela eventos_contato criada.


In [10]:
#Domínios da simulação

NOMES = np.array([
    "João", "José", "Carlos", "Marcos", "Paulo", "Lucas",
    "Mateus", "Rafael", "Fernando", "Bruno", "Ricardo",
    "André", "Gustavo", "Roberto", "Eduardo", "Henrique",
    "Igor", "Daniel", "Felipe", "Luiz", "Marcelo"
])

SOBRENOMES = np.array([
    "Silva", "Santos", "Oliveira", "Souza", "Pereira",
    "Costa", "Rodrigues", "Almeida", "Nascimento",
    "Lima", "Araújo", "Fernandes", "Carvalho", "Gomes",
    "Martins", "Barbosa", "Ribeiro", "Moura"
])

DDDS = np.array([
    11, 12, 13, 14, 15, 16, 17, 18, 19,
    34, 35, 43, 44, 62, 64, 65, 66, 67
])

In [11]:
#Gerar nomes sintéticos

def gerar_nomes(quantidade: int, gerador: np.random.Generator) -> np.ndarray:
    primeiros_nomes = gerador.choice(NOMES, size=quantidade)
    sobrenomes = gerador.choice(SOBRENOMES, size=quantidade)

    nomes_completos = np.char.add(
        np.char.add(primeiros_nomes.astype(str), " "),
        sobrenomes.astype(str)
    )

    return nomes_completos

In [12]:
#Gerar telefones únicos
#Essa função usa o id_cliente para garantir que o telefone não repita.

def gerar_telefones_unicos(
    ids_clientes: np.ndarray,
    gerador: np.random.Generator
) -> list[str]:
    ddds = gerador.choice(DDDS, size=len(ids_clientes))

    telefones = [
        f"55{ddd}9{id_cliente:08d}"
        for ddd, id_cliente in zip(ddds, ids_clientes)
    ]

    return telefones

In [13]:
#Gerar culturas e estágios

def gerar_perfil_agricola(
    quantidade: int,
    gerador: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
    
    culturas = gerador.choice(
        ["Cana", "Soja", "Milho"],
        size=quantidade,
        p=[0.45, 0.35, 0.20]
    )

    estagios = gerador.choice(
        ["Plantio", "Desenvolvimento", "Safra", "Entresafra"],
        size=quantidade,
        p=[0.25, 0.30, 0.25, 0.20]
    )

    return culturas, estagios

In [14]:
#Gerar status dos leads

def gerar_status_leads(
    quantidade: int,
    gerador: np.random.Generator
) -> np.ndarray:
    
    status = gerador.choice(
        [
            "Disponível",
            "Em Cooldown",
            "Fila Prioritária",
            "Em Atendimento",
            "Convertido"
        ],
        size=quantidade,
        p=[0.72, 0.12, 0.06, 0.04, 0.06]
    )

    return status

In [15]:
#Calcular score inicial

def calcular_score_inicial(
    culturas: np.ndarray,
    estagios: np.ndarray,
    status: np.ndarray,
    gerador: np.random.Generator
) -> np.ndarray:
    
    score_base = gerador.normal(
        loc=50,
        scale=15,
        size=len(culturas)
    )

    score_base = np.clip(score_base, 1, 100)

    multiplicador_estagio = np.select(
        [
            estagios == "Plantio",
            estagios == "Safra",
            estagios == "Desenvolvimento",
            estagios == "Entresafra"
        ],
        [
            1.50,
            1.35,
            1.00,
            0.70
        ],
        default=1.00
    )

    multiplicador_cultura = np.select(
        [
            culturas == "Cana",
            culturas == "Soja",
            culturas == "Milho"
        ],
        [
            1.10,
            1.00,
            0.95
        ],
        default=1.00
    )

    multiplicador_status = np.select(
        [
            status == "Fila Prioritária",
            status == "Disponível",
            status == "Em Atendimento",
            status == "Em Cooldown",
            status == "Convertido"
        ],
        [
            1.80,
            1.00,
            0.30,
            0.15,
            0.00
        ],
        default=1.00
    )

    score_final = score_base * multiplicador_estagio * multiplicador_cultura * multiplicador_status
    score_final = np.clip(score_final, 0, 250)

    return np.round(score_final, 2)

In [16]:
#Gerar datas operacionais

def gerar_datas_operacionais(
    status: np.ndarray,
    gerador: np.random.Generator
) -> tuple[pd.Series, pd.Series]:
    
    quantidade = len(status)
    agora = pd.Timestamp.now().floor("s")

    minutos_desde_ultimo_contato = gerador.integers(
        low=0,
        high=60 * 24 * 30,
        size=quantidade
    )

    ultimo_contato = agora - pd.to_timedelta(
        minutos_desde_ultimo_contato,
        unit="m"
    )

    ultimo_contato = pd.Series(ultimo_contato).dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    sem_contato_anterior = gerador.random(quantidade) < 0.18
    ultimo_contato.loc[sem_contato_anterior] = None

    cooldown_ate = pd.Series([None] * quantidade, dtype="object")

    mascara_cooldown = status == "Em Cooldown"

    horas_cooldown = gerador.integers(
        low=1,
        high=48,
        size=mascara_cooldown.sum()
    )

    cooldown_ate.loc[mascara_cooldown] = (
        agora + pd.to_timedelta(horas_cooldown, unit="h")
    ).strftime("%Y-%m-%d %H:%M:%S")

    mascara_convertido = status == "Convertido"

    cooldown_ate.loc[mascara_convertido] = (
        agora + pd.Timedelta(days=30)
    ).strftime("%Y-%m-%d %H:%M:%S")

    return ultimo_contato, cooldown_ate

In [17]:
#Montar um lote de leads

def gerar_lote_leads(
    id_inicial: int,
    quantidade_lote: int,
    gerador: np.random.Generator
) -> pd.DataFrame:
    
    ids_clientes = np.arange(
        id_inicial,
        id_inicial + quantidade_lote
    )

    nomes = gerar_nomes(
        quantidade=quantidade_lote,
        gerador=gerador
    )

    telefones = gerar_telefones_unicos(
        ids_clientes=ids_clientes,
        gerador=gerador
    )

    culturas, estagios = gerar_perfil_agricola(
        quantidade=quantidade_lote,
        gerador=gerador
    )

    status = gerar_status_leads(
        quantidade=quantidade_lote,
        gerador=gerador
    )

    ultimo_contato, cooldown_ate = gerar_datas_operacionais(
        status=status,
        gerador=gerador
    )

    score_prioridade = calcular_score_inicial(
        culturas=culturas,
        estagios=estagios,
        status=status,
        gerador=gerador
    )

    dados_lote = pd.DataFrame({
        "id_cliente": ids_clientes,
        "nome": nomes,
        "telefone": telefones,
        "cultura": culturas,
        "estagio_atual": estagios,
        "status_atual": status,
        "ultimo_contato": ultimo_contato,
        "cooldown_ate": cooldown_ate,
        "score_prioridade": score_prioridade
    })

    return dados_lote

In [18]:
#Testar geração de um pequeno lote

amostra_teste = gerar_lote_leads(
    id_inicial=1,
    quantidade_lote=10,
    gerador=gerador
)

amostra_teste

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,1,José Lima,5534900000001,Cana,Plantio,Disponível,2026-06-12 21:11:55,None,106.46
1,2,Igor Moura,5517900000002,Soja,Desenvolvimento,Disponível,NaN,None,24.76
2,3,Roberto Gomes,5514900000003,Cana,Plantio,Disponível,2026-05-31 16:31:55,None,74.21
3,4,Bruno Gomes,5566900000004,Milho,Plantio,Disponível,2026-06-14 07:33:55,None,74.73
4,5,Bruno Carvalho,5564900000005,Soja,Safra,Disponível,2026-05-29 23:55:55,None,79.37
5,6,Felipe Martins,5543900000006,Soja,Safra,Disponível,2026-06-17 06:57:55,None,81.90
6,7,José Lima,5518900000007,Cana,Entresafra,Em Cooldown,2026-06-18 18:04:55,2026-06-26 19:32:55,7.15
7,8,Eduardo Oliveira,5564900000008,Milho,Desenvolvimento,Disponível,2026-06-05 11:09:55,None,42.53
8,9,Paulo Barbosa,5534900000009,Milho,Desenvolvimento,Disponível,NaN,None,40.91
9,10,José Nascimento,5518900000010,Soja,Desenvolvimento,Em Cooldown,NaN,2026-06-27 06:32:55,9.43


In [19]:
#Inserir dados em lotes no banco

def inserir_leads_em_lotes(
    conexao: sqlite3.Connection,
    quantidade_total: int,
    tamanho_lote: int,
    gerador: np.random.Generator
) -> None:
    
    inicio = time.time()

    total_inserido = 0
    id_atual = 1

    while total_inserido < quantidade_total:
        quantidade_restante = quantidade_total - total_inserido
        quantidade_lote_atual = min(tamanho_lote, quantidade_restante)

        print(
            f"Gerando registros "
            f"{total_inserido + 1:,} até "
            f"{total_inserido + quantidade_lote_atual:,}"
        )

        dados_lote = gerar_lote_leads(
            id_inicial=id_atual,
            quantidade_lote=quantidade_lote_atual,
            gerador=gerador
        )

        dados_lote.to_sql(
            name="leads",
            con=conexao,
            if_exists="append",
            index=False,
            chunksize=50_000
        )

        total_inserido += quantidade_lote_atual
        id_atual += quantidade_lote_atual

        print(f"Total inserido: {total_inserido:,}")

    conexao.commit()

    tempo_total = time.time() - inicio

    print("\nCarga concluída.")
    print(f"Total inserido: {total_inserido:,}")
    print(f"Tempo total: {tempo_total:.2f} segundos")

In [20]:
#Executar carga dos leads

inserir_leads_em_lotes(
    conexao=conexao,
    quantidade_total=QUANTIDADE_LEADS,
    tamanho_lote=TAMANHO_LOTE,
    gerador=gerador
)

Gerando registros 1 até 100,000
Total inserido: 100,000
Gerando registros 100,001 até 200,000
Total inserido: 200,000
Gerando registros 200,001 até 300,000
Total inserido: 300,000
Gerando registros 300,001 até 400,000
Total inserido: 400,000
Gerando registros 400,001 até 500,000
Total inserido: 500,000
Gerando registros 500,001 até 600,000
Total inserido: 600,000
Gerando registros 600,001 até 700,000
Total inserido: 700,000
Gerando registros 700,001 até 800,000
Total inserido: 800,000
Gerando registros 800,001 até 900,000
Total inserido: 900,000
Gerando registros 900,001 até 950,000
Total inserido: 950,000

Carga concluída.
Total inserido: 950,000
Tempo total: 12.77 segundos


In [21]:
#Criar lista de índices

indices_sql = [
    """
    CREATE INDEX IF NOT EXISTS idx_leads_status_score
    ON leads (status_atual, score_prioridade DESC);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_status_cooldown
    ON leads (status_atual, cooldown_ate);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_cultura_estagio
    ON leads (cultura, estagio_atual);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_cooldown_ate
    ON leads (cooldown_ate);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_telefone
    ON leads (telefone);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_disponiveis_score
    ON leads (score_prioridade DESC)
    WHERE status_atual = 'Disponível';
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_fila_prioritaria_score
    ON leads (score_prioridade DESC)
    WHERE status_atual = 'Fila Prioritária';
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_leads_cooldown_expiracao
    ON leads (cooldown_ate)
    WHERE status_atual = 'Em Cooldown';
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_eventos_cliente_data
    ON eventos_contato (id_cliente, data_evento);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_eventos_canal_resultado
    ON eventos_contato (canal, resultado);
    """
]

In [22]:
#Executar criação dos índices

def criar_indices(conexao: sqlite3.Connection, indices: list[str]) -> None:
    for indice in indices:
        conexao.execute(indice)

    conexao.commit()

    conexao.execute("ANALYZE;")
    conexao.execute("PRAGMA optimize;")
    conexao.commit()

    print("Índices criados e banco otimizado.")

In [23]:
criar_indices(conexao, indices_sql)

Índices criados e banco otimizado.


In [24]:
#Validar total de registros

consulta_total = """
SELECT COUNT(*) AS total_leads
FROM leads;
"""

pd.read_sql_query(consulta_total, conexao)

,total_leads
0,950000


In [25]:
#Validar distribuição por status

consulta_status = """
SELECT
    status_atual,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM leads), 2) AS percentual
FROM leads
GROUP BY status_atual
ORDER BY quantidade DESC;
"""

pd.read_sql_query(consulta_status, conexao)

,status_atual,quantidade,percentual
0,Disponível,683811,71.98
1,Em Cooldown,114146,12.02
2,Fila Prioritária,57024,6.00
3,Convertido,56814,5.98
4,Em Atendimento,38205,4.02


In [26]:
#Validar cultura e estágio

consulta_cultura_estagio = """
SELECT
    cultura,
    estagio_atual,
    COUNT(*) AS quantidade,
    ROUND(AVG(score_prioridade), 2) AS score_medio
FROM leads
GROUP BY cultura, estagio_atual
ORDER BY cultura, score_medio DESC;
"""

pd.read_sql_query(consulta_cultura_estagio, conexao)

,cultura,estagio_atual,quantidade,score_medio
0,Cana,Plantio,107413,71.03
1,Cana,Safra,106927,63.73
2,Cana,Desenvolvimento,128022,47.23
3,Cana,Entresafra,85903,32.97
4,Milho,Plantio,47063,61.14
5,Milho,Safra,47755,54.91
6,Milho,Desenvolvimento,57127,40.85
7,Milho,Entresafra,38045,28.65
8,Soja,Plantio,82801,64.19
9,Soja,Safra,82859,57.81


In [27]:
#Testar busca dos próximos leads para robô

consulta_proximos_leads_robo = """
SELECT
    id_cliente,
    nome,
    telefone,
    cultura,
    estagio_atual,
    status_atual,
    ultimo_contato,
    cooldown_ate,
    score_prioridade
FROM leads
WHERE
    status_atual = 'Disponível'
    OR (
        status_atual = 'Em Cooldown'
        AND cooldown_ate <= datetime('now')
    )
ORDER BY score_prioridade DESC
LIMIT 20;
"""

pd.read_sql_query(consulta_proximos_leads_robo, conexao)

,id_cliente,nome,telefone,cultura,estagio_atual,status_atual,ultimo_contato,cooldown_ate,score_prioridade
0,11676,Mateus Souza,5514900011676,Cana,Plantio,Disponível,2026-05-29 01:38:46,None,165.0
1,17515,Ricardo Rodrigues,5518900017515,Cana,Plantio,Disponível,2026-06-01 23:43:46,None,165.0
2,69175,Roberto Araújo,5565900069175,Cana,Plantio,Disponível,2026-05-31 10:34:46,None,165.0
3,85793,Marcos Santos,5518900085793,Cana,Plantio,Disponível,2026-06-01 22:39:46,None,165.0
4,121582,Paulo Silva,5544900121582,Cana,Plantio,Disponível,2026-06-05 08:04:48,None,165.0
5,147144,Rafael Santos,5515900147144,Cana,Plantio,Disponível,2026-06-19 16:04:48,None,165.0
6,237713,Rafael Fernandes,5544900237713,Cana,Plantio,Disponível,2026-05-30 15:16:49,None,165.0
7,260363,José Rodrigues,5512900260363,Cana,Plantio,Disponível,2026-06-20 08:41:49,None,165.0
8,339482,Daniel Carvalho,5516900339482,Cana,Plantio,Disponível,2026-06-03 00:25:50,None,165.0
9,350301,José Silva,5516900350301,Cana,Plantio,Disponível,NaN,None,165.0


In [28]:
#Verificar se o índice está sendo usado

consulta_plano_execucao = """
EXPLAIN QUERY PLAN
SELECT
    id_cliente,
    nome,
    telefone,
    cultura,
    estagio_atual,
    status_atual,
    score_prioridade
FROM leads
WHERE status_atual = 'Disponível'
ORDER BY score_prioridade DESC
LIMIT 20;
"""

pd.read_sql_query(consulta_plano_execucao, conexao)

,id,parent,notused,detail
0,5,0,199,SEARCH leads USING INDEX idx_leads_status_scor...


In [30]:
#Fechar conexão

conexao.close()
print("Conexão encerrada.")

Conexão encerrada.
